In [3]:
import importlib
from snowflake.snowpark.functions import col,trim,split,lit
from snowflake.snowpark.functions import col, sum as _sum, when, is_null
from snowflake.snowpark import functions as F
import sys 
sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
from snowflake.snowpark.session import Session
import pandas as pd
import numpy as np

In [ ]:
Index(['DATES', 'PARENT_DEALER_CODE', 'SKU', 'PREDICTED_SALES_SKU_TFT',
       'FLOOR_PREDICTION_TFT', 'CEIL_PREDICTION_TFT', 'ROUND_PREDICTIONS_TFT',
       'PREDICTED_SALES_SKU_SNOWFLAKE', 'FLOOR_PREDICTION_SNOWFLAKE',
       'CEIL_PREDICTION_SNOWFLAKE', 'ROUND_PREDICTIONS_SNOWFLAKE'],
      dtype='object')

In [4]:
session = Session.builder.configs(snowflake_conn_prop).create()

In [3]:
#Read the forecast from Snowflake
snowflake_forecast = session.sql("SELECT * FROM MOP_DATABASE.SOQ.FORECAST_JAN_26_TO_APRIL_26_VIEW").to_pandas()

In [4]:
snowflake_forecast.head()

,DATES,SERIES,PREDICTED_SALES
0,2026-01-01,"""10364<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE""",14.438978
1,2026-02-01,"""10364<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE""",22.940716
2,2026-03-01,"""10364<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE""",21.099181
3,2026-04-01,"""10364<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLUE""",34.711104
4,2026-01-01,"""10097<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAV...",8.725526


In [5]:
snowflake_forecast["SERIES"] = snowflake_forecast["SERIES"].apply(lambda x:x.replace('"',''))

In [6]:
#Jan 26 to Mar 26 predictions using TFT
jan_to_mar = pd.read_csv(r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Model\Jan_26_to_Mar_26_prediction.csv")

apr_to_june = pd.read_csv(r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Model\April_26_to_June_26.csv")

In [7]:
jan_to_mar.drop(columns=jan_to_mar.columns.tolist()[0],inplace=True)
jan_to_mar

,MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,ACTUAL_SALES,PREDICTED_SALES
0,2026-01-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.0,2.76
1,2026-02-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.0,3.03
2,2026-03-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.0,3.21
3,2026-01-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,0.0,7.38
4,2026-02-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,0.0,6.72
...,...,...,...,...
105304,2026-02-01,17062<>XPULSE<>DISC<>SELF<>SPOKE<>BLUE,2.0,5.00
105305,2026-03-01,17062<>XPULSE<>DISC<>SELF<>SPOKE<>BLUE,1.0,7.19
105306,2026-01-01,17071<>XPULSE<>DISC<>SELF<>SPOKE<>BLUE,0.0,12.77
105307,2026-02-01,17071<>XPULSE<>DISC<>SELF<>SPOKE<>BLUE,0.0,10.57


In [8]:
apr_to_june.head()

,MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,PREDICTED_SALES
0,2026-04-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,3.48
1,2026-05-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,3.48
2,2026-06-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.02
3,2026-04-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,8.67
4,2026-05-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,7.43


In [9]:
snowflake_forecast.rename(columns={"DATES":"MONTH_OF_SALE","SERIES":"PARENT_DEALER_CODE_MODEL_FAMILY"},inplace=True)

In [10]:
#Ensuring all the series available in jan_to_mar data are also available in apr_to_june

unique_series_jan_to_mar = set(jan_to_mar["PARENT_DEALER_CODE_MODEL_FAMILY"].unique())

unique_series_apr_to_june = set(apr_to_june["PARENT_DEALER_CODE_MODEL_FAMILY"].unique())

unique_series_snowflake_data = set(snowflake_forecast["PARENT_DEALER_CODE_MODEL_FAMILY"].unique())

In [11]:
# 1. Calculate the directional differences (Finding elements that violate your rules)
# Rule A: Everything in apr_to_june must be inside jan_to_mar
unanchored_apr_june = unique_series_apr_to_june - unique_series_jan_to_mar

# Rule B: Everything in snowflake_forecast must be inside jan_to_mar
snowflake_missing_from_jan_mar = unique_series_snowflake_data - unique_series_jan_to_mar

# Rule C: Everything in snowflake_forecast must be inside apr_to_june
snowflake_missing_from_apr_june = unique_series_snowflake_data - unique_series_apr_to_june


# ==========================================
# 2. RUN STRICT SUBSET VALIDATION REPORT
# ==========================================
print("--- STRICT HORIZON SUBSET INTEGRITY REPORT ---")

# Check A: apr_to_june ⊆ jan_to_mar
if unique_series_apr_to_june.issubset(unique_series_jan_to_mar):
    print("✅ PASS: All series in 'apr_to_june' are fully covered inside 'jan_to_mar'.")
else:
    print(f"❌ FAIL: Found {len(unanchored_apr_june)} series in 'apr_to_june' that DO NOT exist in 'jan_to_mar'!")
    print(f"   -> Samples: {list(unanchored_apr_june)[:5]}")

# Check B & C: snowflake_forecast ⊆ (jan_to_mar ∩ apr_to_june)
snowflake_is_safe = unique_series_snowflake_data.issubset(unique_series_jan_to_mar) and \
                    unique_series_snowflake_data.issubset(unique_series_apr_to_june)

if snowflake_is_safe:
    print("✅ PASS: All series in 'snowflake_forecast' are perfectly covered by both 'jan_to_mar' and 'apr_to_june'.")
else:
    print("❌ FAIL: 'snowflake_forecast' contains series that are missing from your covariate sets!")
    if snowflake_missing_from_jan_mar:
        print(f"   -> Missing from 'jan_to_mar' ({len(snowflake_missing_from_jan_mar)}): {list(snowflake_missing_from_jan_mar)[:5]}")
    if snowflake_missing_from_apr_june:
        print(f"   -> Missing from 'apr_to_june' ({len(snowflake_missing_from_apr_june)}): {list(snowflake_missing_from_apr_june)[:5]}")

--- STRICT HORIZON SUBSET INTEGRITY REPORT ---
✅ PASS: All series in 'apr_to_june' are fully covered inside 'jan_to_mar'.
✅ PASS: All series in 'snowflake_forecast' are perfectly covered by both 'jan_to_mar' and 'apr_to_june'.


In [12]:
#concatenate apr_to_june and jan_to_mar data

apr_to_june.head()

,MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,PREDICTED_SALES
0,2026-04-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,3.48
1,2026-05-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,3.48
2,2026-06-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.02
3,2026-04-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,8.67
4,2026-05-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,7.43


In [13]:
jan_to_mar.head()

,MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,ACTUAL_SALES,PREDICTED_SALES
0,2026-01-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.0,2.76
1,2026-02-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.0,3.03
2,2026-03-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,0.0,3.21
3,2026-01-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,0.0,7.38
4,2026-02-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,0.0,6.72


In [14]:
jan_to_mar.drop(columns=["ACTUAL_SALES"],inplace=True)

In [15]:
full_tft_prediction = pd.concat([jan_to_mar,apr_to_june],axis=0,ignore_index=True)

In [16]:
full_tft_prediction.head()

,MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,PREDICTED_SALES
0,2026-01-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,2.76
1,2026-02-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,3.03
2,2026-03-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,3.21
3,2026-01-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,7.38
4,2026-02-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,6.72


In [17]:
full_tft_prediction.reset_index(inplace=True)

In [18]:
snowflake_forecast["PARENT_DEALER_CODE_MODEL_FAMILY"].nunique()

35103

In [19]:
snowflake_forecast.rename(columns={"PREDICTED_SALES":"PREDICTED_SALES_SNOWFLAKE"},inplace=True)

In [20]:
snowflake_forecast.shape

(140412, 3)

In [21]:
full_tft_prediction["MONTH_OF_SALE"] = pd.to_datetime(full_tft_prediction["MONTH_OF_SALE"])
full_tft_prediction.rename(columns={"PREDICTED_SALES":"PREDICTED_SALES_TFT"},inplace=True)

In [22]:
#Join with snowflake_prediction 
snowflake_and_tft_prediction = pd.merge(left=full_tft_prediction,right=snowflake_forecast,on=["PARENT_DEALER_CODE_MODEL_FAMILY","MONTH_OF_SALE"])

snowflake_and_tft_prediction.shape

(140412, 5)

In [23]:
snowflake_and_tft_prediction.head()

,index,MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,PREDICTED_SALES_TFT,PREDICTED_SALES_SNOWFLAKE
0,0,2026-01-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,2.76,2.495567
1,1,2026-02-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,3.03,2.691133
2,2,2026-03-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>BLACK,3.21,3.087983
3,3,2026-01-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,7.38,7.382140
4,4,2026-02-01,10001<>DESTINI<>DRUM<>SELF<>CAST<>WHITE,6.72,8.182140


In [24]:
#read the actual data
actual_data = session.sql("""
SELECT MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,NET_SALES
        FROM MOP_DATABASE.SOQ.TRAIN_AND_TEST_DATA_FOR_TFT
        WHERE MONTH_OF_SALE BETWEEN '2026-01-01' AND '2026-04-01'
  AND PARENT_DEALER_CODE_MODEL_FAMILY IN (
      SELECT VALID_PARENT_DEALER_CODE_MODEL_FAMILY_COMBINATIONS
      FROM MOP_DATABASE.SOQ.VALID_COMBINATIONS_USED_FOR_TFT
  )
""").to_pandas()

In [25]:
actual_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140412 entries, 0 to 140411
Data columns (total 3 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   MONTH_OF_SALE                    140412 non-null  object 
 1   PARENT_DEALER_CODE_MODEL_FAMILY  140412 non-null  object 
 2   NET_SALES                        140412 non-null  float64
dtypes: float64(1), object(2)
memory usage: 3.2+ MB


In [26]:
actual_data["MONTH_OF_SALE"] = pd.to_datetime(actual_data["MONTH_OF_SALE"])

In [27]:
actual_data.head()

,MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,NET_SALES
0,2026-02-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,1.0
1,2026-01-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,5.0
2,2026-03-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,2.0
3,2026-04-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,3.0
4,2026-02-01,12047<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,2.0


In [28]:
snowflake_and_tft_prediction_with_actual_sales = pd.merge(left=actual_data,right=snowflake_and_tft_prediction,on=["PARENT_DEALER_CODE_MODEL_FAMILY","MONTH_OF_SALE"])

In [29]:
snowflake_and_tft_prediction_with_actual_sales.shape

(140412, 6)

In [30]:
snowflake_and_tft_prediction_with_actual_sales.head()

,MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,NET_SALES,index,PREDICTED_SALES_TFT,PREDICTED_SALES_SNOWFLAKE
0,2026-02-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,1.0,90448,3.27,0.979130
1,2026-01-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,5.0,90447,3.46,0.655424
2,2026-03-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,2.0,90449,3.29,0.917071
3,2026-04-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,3.0,195756,2.93,1.926597
4,2026-02-01,12047<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,2.0,91627,5.31,4.406495


In [31]:
snowflake_and_tft_prediction_with_actual_sales.drop(columns=['index'],inplace=True)

In [32]:
df = snowflake_and_tft_prediction_with_actual_sales.copy()

In [33]:
df.rename(columns={"NET_SALES":"ACTUAL_SALES"},inplace=True)

In [48]:
data = df.copy()
data = data.loc[data["ACTUAL_SALES"]>10,:]

In [34]:
import pandas as pd
import numpy as np

# ── Guard against zero actuals ───────────────────────────────────────────────
df = df[df['ACTUAL_SALES'] != 0].copy()  # drop zero-actual rows before MAPE
# If you want to keep them but flag them instead, swap the line above with:
# df['ZERO_ACTUAL_FLAG'] = df['ACTUAL_SALES'] == 0

# ── Row-level absolute percentage errors ─────────────────────────────────────
df['APE_TFT']       = (abs(df['ACTUAL_SALES'] - df['PREDICTED_SALES_TFT'])       / abs(df['ACTUAL_SALES'])) * 100
df['APE_SNOWFLAKE'] = (abs(df['ACTUAL_SALES'] - df['PREDICTED_SALES_SNOWFLAKE']) / abs(df['ACTUAL_SALES'])) * 100

# ── Aggregate by month ────────────────────────────────────────────────────────
monthly_comparison = (
    df.groupby('MONTH_OF_SALE')
    .agg(
        SERIES_COUNT          = ('PARENT_DEALER_CODE_MODEL_FAMILY', 'nunique'),
        MAPE_TFT              = ('APE_TFT',       'mean'),
        MAPE_SNOWFLAKE        = ('APE_SNOWFLAKE',  'mean'),
        MEDIAN_APE_TFT        = ('APE_TFT',       'median'),   # median is more robust to outlier series
        MEDIAN_APE_SNOWFLAKE  = ('APE_SNOWFLAKE',  'median'),
    )
    .reset_index()
)

# ── Which model wins each month ───────────────────────────────────────────────
monthly_comparison['BETTER_MODEL'] = np.where(
    monthly_comparison['MAPE_TFT'] < monthly_comparison['MAPE_SNOWFLAKE'],
    'TFT', 'SNOWFLAKE'
)

monthly_comparison['MAPE_DIFFERENCE_PP'] = (
    monthly_comparison['MAPE_SNOWFLAKE'] - monthly_comparison['MAPE_TFT']
)  # positive = TFT is better by that many percentage points

# ── Round for readability ─────────────────────────────────────────────────────
round_cols = ['MAPE_TFT', 'MAPE_SNOWFLAKE', 'MEDIAN_APE_TFT', 'MEDIAN_APE_SNOWFLAKE', 'MAPE_DIFFERENCE_PP']
monthly_comparison[round_cols] = monthly_comparison[round_cols].round(2)

monthly_comparison = monthly_comparison.sort_values('MONTH_OF_SALE').reset_index(drop=True)

# ── Summary line ──────────────────────────────────────────────────────────────
print(f"Months evaluated  : {monthly_comparison['MONTH_OF_SALE'].nunique()}")
print(f"TFT wins          : {(monthly_comparison['BETTER_MODEL'] == 'TFT').sum()} months")
print(f"Snowflake wins    : {(monthly_comparison['BETTER_MODEL'] == 'SNOWFLAKE').sum()} months")
print(f"Avg MAPE TFT      : {monthly_comparison['MAPE_TFT'].mean():.2f}%")
print(f"Avg MAPE Snowflake: {monthly_comparison['MAPE_SNOWFLAKE'].mean():.2f}%")

monthly_comparison

Months evaluated  : 4
TFT wins          : 4 months
Snowflake wins    : 0 months
Avg MAPE TFT      : 87.82%
Avg MAPE Snowflake: 96.12%


,MONTH_OF_SALE,SERIES_COUNT,MAPE_TFT,MAPE_SNOWFLAKE,MEDIAN_APE_TFT,MEDIAN_APE_SNOWFLAKE,BETTER_MODEL,MAPE_DIFFERENCE_PP
0,2026-01-01,22976,84.11,91.17,41.77,60.07,TFT,7.05
1,2026-02-01,22530,89.20,96.06,43.00,62.37,TFT,6.86
2,2026-03-01,23214,94.35,94.50,44.20,60.80,TFT,0.14
3,2026-04-01,22828,83.63,102.76,43.00,67.10,TFT,19.13


In [45]:
df.loc[df["ACTUAL_SALES"]<5,'PARENT_DEALER_CODE_MODEL_FAMILY'].nunique()

19538

In [54]:
data = df.loc[df["ACTUAL_SALES"]>20,:]

In [55]:
import pandas as pd
import numpy as np

# ── Guard against zero actuals ───────────────────────────────────────────────
data = data[data['ACTUAL_SALES'] != 0].copy()  # drop zero-actual rows before MAPE
# If you want to keep them but flag them instead, swap the line above with:
# data['ZERO_ACTUAL_FLAG'] = data['ACTUAL_SALES'] == 0

# ── Row-level absolute percentage errors ─────────────────────────────────────
data['APE_TFT']       = (abs(data['ACTUAL_SALES'] - data['PREDICTED_SALES_TFT'])       / abs(data['ACTUAL_SALES'])) * 100
data['APE_SNOWFLAKE'] = (abs(data['ACTUAL_SALES'] - data['PREDICTED_SALES_SNOWFLAKE']) / abs(data['ACTUAL_SALES'])) * 100

# ── Aggregate by month ────────────────────────────────────────────────────────
monthly_comparison = (
    data.groupby('MONTH_OF_SALE')
    .agg(
        SERIES_COUNT          = ('PARENT_DEALER_CODE_MODEL_FAMILY', 'nunique'),
        MAPE_TFT              = ('APE_TFT',       'mean'),
        MAPE_SNOWFLAKE        = ('APE_SNOWFLAKE',  'mean'),
        MEDIAN_APE_TFT        = ('APE_TFT',       'median'),   # median is more robust to outlier series
        MEDIAN_APE_SNOWFLAKE  = ('APE_SNOWFLAKE',  'median'),
    )
    .reset_index()
)

# ── Which model wins each month ───────────────────────────────────────────────
monthly_comparison['BETTER_MODEL'] = np.where(
    monthly_comparison['MAPE_TFT'] < monthly_comparison['MAPE_SNOWFLAKE'],
    'TFT', 'SNOWFLAKE'
)

monthly_comparison['MAPE_DIFFERENCE_PP'] = (
    monthly_comparison['MAPE_SNOWFLAKE'] - monthly_comparison['MAPE_TFT']
)  # positive = TFT is better by that many percentage points

# ── Round for readability ─────────────────────────────────────────────────────
round_cols = ['MAPE_TFT', 'MAPE_SNOWFLAKE', 'MEDIAN_APE_TFT', 'MEDIAN_APE_SNOWFLAKE', 'MAPE_DIFFERENCE_PP']
monthly_comparison[round_cols] = monthly_comparison[round_cols].round(2)

monthly_comparison = monthly_comparison.sort_values('MONTH_OF_SALE').reset_index(drop=True)

# ── Summary line ──────────────────────────────────────────────────────────────
print(f"Months evaluated  : {monthly_comparison['MONTH_OF_SALE'].nunique()}")
print(f"TFT wins          : {(monthly_comparison['BETTER_MODEL'] == 'TFT').sum()} months")
print(f"Snowflake wins    : {(monthly_comparison['BETTER_MODEL'] == 'SNOWFLAKE').sum()} months")
print(f"Avg MAPE TFT      : {monthly_comparison['MAPE_TFT'].mean():.2f}%")
print(f"Avg MAPE Snowflake: {monthly_comparison['MAPE_SNOWFLAKE'].mean():.2f}%")

monthly_comparison

Months evaluated  : 4
TFT wins          : 4 months
Snowflake wins    : 0 months
Avg MAPE TFT      : 33.17%
Avg MAPE Snowflake: 43.15%


,MONTH_OF_SALE,SERIES_COUNT,MAPE_TFT,MAPE_SNOWFLAKE,MEDIAN_APE_TFT,MEDIAN_APE_SNOWFLAKE,BETTER_MODEL,MAPE_DIFFERENCE_PP
0,2026-01-01,4369,30.85,42.10,25.90,33.93,TFT,11.26
1,2026-02-01,4143,31.98,42.70,27.09,34.38,TFT,10.71
2,2026-03-01,4805,34.15,41.36,28.23,33.33,TFT,7.22
3,2026-04-01,4817,35.71,46.42,29.14,36.73,TFT,10.71


In [35]:
df.head()

,MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,ACTUAL_SALES,PREDICTED_SALES_TFT,PREDICTED_SALES_SNOWFLAKE,APE_TFT,APE_SNOWFLAKE
0,2026-02-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,1.0,3.27,0.979130,227.000000,2.087031
1,2026-01-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,5.0,3.46,0.655424,30.800000,86.891528
2,2026-03-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,2.0,3.29,0.917071,64.500000,54.146470
3,2026-04-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,3.0,2.93,1.926597,2.333333,35.780096
4,2026-02-01,12047<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,2.0,5.31,4.406495,165.500000,120.324725


In [36]:
df.drop(columns=["APE_TFT","APE_SNOWFLAKE"],inplace=True)

In [37]:
df.to_csv(r"Prediction_comparison_snowflake_and_tft.csv",index=False)

In [56]:
df.head()

,MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,ACTUAL_SALES,PREDICTED_SALES_TFT,PREDICTED_SALES_SNOWFLAKE
0,2026-02-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,1.0,3.27,1.0
1,2026-01-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,5.0,3.46,1.0
2,2026-03-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,2.0,3.29,1.0
3,2026-04-01,12015<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,3.0,2.93,2.0
4,2026-02-01,12047<>PASSION<>DRUM<>SELF<>CASTI<>BLACK HEAVY...,2.0,5.31,5.0


### Calculating the monthly retail numbers

In [13]:
# all_series_data = session.sql("""
#     SELECT 
#         TO_CHAR(MONTH_OF_SALE, 'MMMM - YYYY') AS FORMATTED_MONTH,
#         SUM(NET_SALES) AS TOTAL_MONTHLY_SALES
#     FROM MOP_DATABASE.SOQ.TRAIN_AND_TEST_DATA_FOR_TFT
#     GROUP BY DATE_TRUNC('MONTH', MONTH_OF_SALE), TO_CHAR(MONTH_OF_SALE, 'MMMM - YYYY')
#     ORDER BY DATE_TRUNC('MONTH', MONTH_OF_SALE) DESC
# """)

selected_series_data = session.sql("""
SELECT 
        TO_CHAR(MONTH_OF_SALE, 'MMMM - YYYY') AS FORMATTED_MONTH,
        SUM(NET_SALES) AS TOTAL_MONTHLY_SALES
    FROM MOP_DATABASE.SOQ.TRAIN_AND_TEST_DATA_FOR_TFT
    WHERE PARENT_DEALER_CODE_MODEL_FAMILY IN (SELECT VALID_PARENT_DEALER_CODE_MODEL_FAMILY_COMBINATIONS
      FROM MOP_DATABASE.SOQ.VALID_COMBINATIONS_USED_FOR_TFT)
    GROUP BY DATE_TRUNC('MONTH', MONTH_OF_SALE), TO_CHAR(MONTH_OF_SALE, 'MMMM - YYYY')
    ORDER BY DATE_TRUNC('MONTH', MONTH_OF_SALE) DESC
""")

In [14]:
# all_series_data.show()

In [9]:
all_pandas_data=all_series_data.to_pandas()

all_pandas_data.head()

,FORMATTED_MONTH,TOTAL_MONTHLY_SALES
0,April - 2027,0.0
1,March - 2027,0.0
2,February - 2027,0.0
3,January - 2027,0.0
4,December - 2026,0.0


In [10]:
all_pandas_data.to_csv(r"Total_sales.csv",index=False)

In [16]:
selected_series_retail_sales=selected_series_data.to_pandas()
selected_series_retail_sales.to_csv(r"Selected_series_sales.csv",index=False)

In [17]:
filtered_out_series_data = session.sql("""
SELECT 
        TO_CHAR(MONTH_OF_SALE, 'MMMM - YYYY') AS FORMATTED_MONTH,
        SUM(NET_SALES) AS TOTAL_MONTHLY_SALES
    FROM MOP_DATABASE.SOQ.TRAIN_AND_TEST_DATA_FOR_TFT
    WHERE PARENT_DEALER_CODE_MODEL_FAMILY IN (SELECT SERIES_NAME
      FROM MOP_DATABASE.SOQ.INVALID_PARENT_DEALER_COMBINATIONS)
    GROUP BY DATE_TRUNC('MONTH', MONTH_OF_SALE), TO_CHAR(MONTH_OF_SALE, 'MMMM - YYYY')
    ORDER BY DATE_TRUNC('MONTH', MONTH_OF_SALE) DESC
""")

In [18]:
filtered_out_series_retail_sales=filtered_out_series_data.to_pandas()
filtered_out_series_retail_sales.to_csv(r"Filtered_series_sales.csv",index=False)